### Importy

In [1]:
import math
from scipy.stats import norm, t
from typing import Literal, Optional

### Potrzebne funkcje

In [2]:
def no_estymator_wariancji(dane: list[float]) -> float:
    n = len(dane)
    x_bar = sum(dane) / n
    sq_diff = list(map(lambda x: (x - x_bar)**2, dane))
    return sum(sq_diff) / (n - 1)

def hipoteza_sredniej_jedna_proba(
    probka: list[float],
    srednia_h0: float,
    poziom_istotnosci: float,
    odchylenie_populacji: Optional[float] = None,
    typ_hipotezy: Literal["L", "P", "O"] = "O"
) -> bool:
    n = len(probka)
    srednia_proby = sum(probka) / n

    if odchylenie_populacji is not None:
        sigma = odchylenie_populacji
        uzyj_rozkadu_normalnego = True
    else:
        sigma = math.sqrt(no_estymator_wariancji(probka)) if n > 1 else 0.0
        uzyj_rozkadu_normalnego = n > 30

    statystyka = (srednia_proby - srednia_h0) * math.sqrt(n) / sigma

    if uzyj_rozkadu_normalnego:
        if typ_hipotezy == 'L':
            war_kryt = -norm.ppf(1 - poziom_istotnosci)
            return bool(statystyka > war_kryt)
        elif typ_hipotezy == 'P':
            war_kryt = norm.ppf(1 - poziom_istotnosci)
            return bool(statystyka < war_kryt)
        else: # Obustronny
            war_kryt = norm.ppf(1 - poziom_istotnosci / 2)
            return bool(-war_kryt < statystyka < war_kryt)
    else:
        stopien_swobody = n - 1
        if typ_hipotezy == 'L':
            war_kryt = -t.ppf(1 - poziom_istotnosci, stopien_swobody)
            return bool(statystyka > war_kryt)
        elif typ_hipotezy == 'P':
            war_kryt = t.ppf(1 - poziom_istotnosci, stopien_swobody)
            return bool(statystyka < war_kryt)
        else: # Obustronny
            war_kryt = t.ppf(1 - poziom_istotnosci / 2, stopien_swobody)
            return bool(-war_kryt < statystyka < war_kryt)

## Zadanie 1

* $H_0$ - procent butelek wybrakowanych wynosi $p_0 = 0.03$.
* $H_1$ - procent butelek wybrakowanych $< 0.03$.

Obliczenia przeprowadzamy na poziomie istotności $\alpha = 0.05$

Traktujemy próbę $n$ butelek jako zbiór $n$ zmiennych losowych $X_i$ o rozkładzie ***Bernoulliego*** z prawdopodobieństwem $p_0$
* $X_i = 1$, gdy butelka jest wadliwa
* $X_i = 0$, gdy butelka jest dobra

Widać, że średnia z tej próbki $\bar{X} = \frac{1}{n}\sum_{i=1}^nX_i$ jest proporcją wadliwych butelek w próbce $n$ butelek do wszystkich butelek w próbie.

Mamy:

$$
    \mathbb{E}(X_i) = p_0\\
    \mathrm{Var}(X_i) = p_0(1-p_0)\\
    \sigma = \sqrt{p_0(1-p_0)}
$$  

Możemy więc policzyć statystykę testową
$$
    T = \frac{\bar{X} - p_0}{\frac{\sigma}{\sqrt{n}}}
$$

Aby rozstrzygnąć czy zostawiamy hipotezę zerową czy ją drzucamy, korzystamy z fnukcji `hipoteza_sredniej_jedna_proba` z parametrami:

In [3]:
n = 900
wadliwe = 18
p0 = 0.03
X_bar = wadliwe / n
alfa = 0.05
sigma = math.sqrt(p0 * (1 - p0))

BUTELKI = [1] * wadliwe + [0] * (n - wadliwe)

print(f"{n = }")
print(f"{wadliwe = }")
print(f"{p0 = }")
print(f"{X_bar = }")
print(f"{alfa = }")
print(f"{sigma = }")

werdykt = hipoteza_sredniej_jedna_proba(BUTELKI, 
                                        p0,
                                        alfa,
                                        sigma,
                                        "L")

print(f"Utrzymujemy hipotezę zerową: {werdykt}")

n = 900
wadliwe = 18
p0 = 0.03
X_bar = 0.02
alfa = 0.05
sigma = 0.1705872210923198
Utrzymujemy hipotezę zerową: False


## Zadanie 2

* $H_0$ - procent wadliwych urządzeń $\le 0.06$.
* $H_1$ - procent wadliwych urządzeń $> 0.06$

Przyjmujemy najgorszy scenariusz dla hipotey $H_0$, czyli $p_0 = 0.06$. Obliczenia przeprowadzamy na poziomie istotności $\alpha = 0.1$

Traktujemy próbę $n$ urządzeń jako zbiór $n$ zmiennych losowych $X_i$ o rozkładzie ***Bernoulliego*** z prawdopodobieństwem $p_0$
* $X_i = 1$, gdy urządzenie jest wadliwe
* $X_i = 0$, gdy urządzenie jest dobre

Widać, że średnia z tej próbki $\bar{X} = \frac{1}{n}\sum_{i=1}^nX_i$ jest proporcją wadliwych urządzeń w próbce $n$ urządzeń do wszystkich urządzeń w próbie.

Mamy:

$$
    \mathbb{E}(X_i) = p_0\\
    \mathrm{Var}(X_i) = p_0(1-p_0)\\
    \sigma = \sqrt{p_0(1-p_0)}
$$  

Możemy więc policzyć statystykę testową
$$
    T = \frac{\bar{X} - p_0}{\frac{\sigma}{\sqrt{n}}}
$$

Aby rozstrzygnąć czy zostawiamy hipotezę zerową czy ją drzucamy, korzystamy z fnukcji `hipoteza_sredniej_jedna_proba` z parametrami:

In [4]:
n = 15
wadliwe = 3
p0 = 0.06
alfa = 0.1
sigma = math.sqrt(p0 * (1 - p0))
URZADZENIA = [1] * wadliwe + [0] * (n - wadliwe)

werdykt = hipoteza_sredniej_jedna_proba(URZADZENIA,
                                        p0,
                                        alfa,
                                        sigma,
                                        'P')

print(f"Utrzymujemy hipotezę zerową: {werdykt}")

Utrzymujemy hipotezę zerową: False


## Zadanie 4

* $H_0$ - Śruby mają przeciętną średnicę $d_0 = 8,1$.
* $H_1$ - Śruby mają przeciętną średnicę $> d_0$

Obliczenia przeprowadzamy na poziomie istotności $\alpha = 0.01$

Zmienna $X$ oznacza wynik pomiaru średnicy. Wiemy, że ma ona rozkład normalny. Dane jest odchylenie standardowe $\sigma = 0.03$. Wykorzystujemy statystykę:

$$
    T = \frac{\bar{X} - d_0}{\frac{\sigma}{\sqrt{n}}},
$$

Aby zweryfikować hipotezę kożystamy z funkcji `hipoteza_sredniej_jedna_proba`.

In [5]:
POMIARY = [8.235, 8.183, 8.207, 8.156, 8.167, 8.199, 8.122, 8.186, 8.169, 8.199, 8.245 ,8.181, 8.204, 8.219, 8.183, 8.264, 8.205, 8.181, 8.186, 8.224]

n = len(POMIARY)
d0 = 8.1
X_bar = sum(POMIARY) / n
sigma = 0.03
alfa = 0.01

werdykt = hipoteza_sredniej_jedna_proba(POMIARY,
                                        d0,
                                        alfa,
                                        sigma,
                                        'P')

print(f"Utrzymujemy hipotezę zerową: {werdykt}")

Utrzymujemy hipotezę zerową: False


## Zadanie 7

* $H_0$ - Prawdopodobieństwo uszkodzenia lampy wynosi $p_0 = 0.01$
* $H_1$ - $p_0 \ne 0.01$

Obliczenia przeprowadzamy na poziomiqch istotności a) $\alpha = 0.02$ i b) $\alpha = 0.05$ 

Zmienna $X$ oznacza ilość lamp wymienionych w proóbce $n$ lamp. Jest to zmienna o rozkładzie ***dwumianowym*** z prawdopodobieństwem $p_0$. Widać, że $\bar{X}$ to procent lamp potrzebujących wymiany w próbce $n$ lamp. Obliczamy statystykę. Odchylenie populacji dane jest wzorem $\sigma = \sqrt{p_0(1-p_0)}$.

$$
    T = \frac{\bar{X} - p_0}{\frac{\sigma}{\sqrt{n}}},
$$

Aby zweryfikować hipotezę kożystamy z funkcji `hipoteza_sredniej_jedna_proba`.


In [6]:
n = 1024
do_wymiany = 18
p0 = 0.01
sigma = math.sqrt(p0 * (1 - p0))

LAMPY = [1] * do_wymiany + [0] * (n - do_wymiany)
# a)
alfa = 0.02
werdykt = hipoteza_sredniej_jedna_proba(LAMPY,
                                        p0,
                                        alfa,
                                        sigma,
                                        "O")
print(f"a) Utrzymujemy hipotezę zerową: {werdykt}")
# b)
alfa = 0.05
werdykt = hipoteza_sredniej_jedna_proba(LAMPY,
                                        p0,
                                        alfa,
                                        sigma,
                                        "O")
print(f"b) Utrzymujemy hipotezę zerową: {werdykt}")

a) Utrzymujemy hipotezę zerową: False
b) Utrzymujemy hipotezę zerową: False


In [7]:
def poisson(n: int, p: float, k: int) -> float:
    lam = n * p
    return (lam**k) / math.factorial(k) * math.exp(-lam)

n = 100_000
p = 0.001 * 0.1 * 0.5


PXle3 = sum([poisson(n, p, k) for k in range(4)])
print(f"P(X<=3) = {PXle3}")

P(X<=3) = 0.2650259152973617


In [8]:
def CTG(n: int, mu: float, sigma: float, dol: float = -math.inf, gora: float = math.inf) -> float:
    return norm.cdf((gora - n * mu) / (sigma * math.sqrt(n))) - norm.cdf((dol - n * mu) / (sigma * math.sqrt(n)))

n = 1000
mu = 3.5
sigma = math.sqrt(91/6 - 3.5**2)
dol = 3450
gora = 3550

print(f"Prawdopodobieństwo: {CTG(n,mu,sigma,dol,gora)}")

Prawdopodobieństwo: 0.6454605202264987


In [9]:
def przedzial_ufnosci_srednia(dane: list[float], odch_standardowe: float, alfa: float) -> tuple[float, float]:
    n = len(dane)
    x_bar = sum(dane) / n
    r = norm.ppf(1 - alfa / 2) * odch_standardowe / math.sqrt(n)
    return float(x_bar - r), float(x_bar + r)

n = 1000
mu = 3.5
sigma = n * math.sqrt(91/6 - 3.5**2)
alfa = 0.01

DANE = [mu*n] * n

dol, gora = przedzial_ufnosci_srednia(DANE, sigma, alfa)

print(f"Przedział ufności: {[math.floor(dol), math.ceil(gora)]}")

Przedział ufności: [3360, 3640]


In [10]:
n = 400
sigma = 100 * n
alfa = 0.02
KLIENCI = [0] * n
przedzial_ufnosci_srednia(KLIENCI, sigma, alfa)

(-4652.695748081682, 4652.695748081682)

In [11]:
n = 1000
mu = -500/50 - 1000/100 - 4000/400 + 35
sigma = math.sqrt(500**2/50 + 1000**2/100 + 4000**2/400)
norm.cdf((-1000-n*mu)/(sigma*math.sqrt(n)))

np.float64(0.20924611672340993)

In [12]:
alfa = 0.05
(-norm.ppf(alfa) * sigma / mu)**2

np.float64(5952.195599009913)

In [13]:
import scipy

val_a = 0
val_b = 0
for k in range(10, 26):
    val_a += poisson(1000, 0.02, k)

for k in range(10, 26):
    val_b += scipy.stats.binom.pmf(k, 1000, 0.02)
print(val_a, val_b)

0.8828196149737226 0.8853861393943002


In [19]:
lam = 90

result = 0
frac = 4*lam * math.exp(-4 * lam)
for i in range(1, 81):
    frac *= 4*lam / i
print(frac)
# for k in range(80, 111):
#     result += (4 * lam) ** k * math.exp(-4*lam)

7.240379906960687e-69


In [24]:
a = 80
b = 110
lam = 90
norm.cdf((b - lam) / math.sqrt(lam)) - norm.cdf((a - lam)/math.sqrt(lam))

np.float64(0.8365722369182745)